In [9]:
import time
import csv
import random
import hashlib
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ================= 配置区域 =================
# 替换为你要爬取的商品完整链接
TARGET_URL = "https://detail.tmall.com/item.htm?abbucket=17&id=903091730827..."
CSV_FILE = "tmall_sidebar_reviews.csv"

# === 根据你提供的结构定义的 Class 常量 ===
# 查看全部评价按钮
BTN_SHOW_ALL_CLASS = "ShowButton--fMu7HZNs"
# 单条评论容器
COMMENT_ITEM_CLASS = "Comment--H5QmJwe9"
# 评论内容
CONTENT_CLASS = "content--uonoOhaz"
# 元数据（时间/SKU）
META_CLASS = "meta--PLijz6qf"
# 用户名
USER_NAME_CLASS = "userName--KpyzGX2s"

class TmallSidebarSpider:
    def __init__(self):
        self.driver = self._init_driver()
        self.scraped_hashes = set() # 用于去重

    def _init_driver(self):
        options = Options()
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        # 模拟手机/H5环境可能更容易触发这种侧边栏结构，但也兼容PC
        options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36')

        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
            "source": """Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"""
        })
        driver.maximize_window()
        return driver

    def login_phase(self):
        print(">>> 打开登录页，请在 40秒内 扫码登录...")
        self.driver.get("https://login.taobao.com/member/login.jhtml")
        # 等待用户手动登录，直到检测到跳转或时间结束
        time.sleep(10)
        input(">>> 扫码登录成功后，请在此终端按【回车键】继续...")

    def open_sidebar(self):
        """找到并点击[查看全部评价]按钮"""
        print(f">>> 正在访问商品页: {TARGET_URL}")
        self.driver.get(TARGET_URL)
        time.sleep(3)

        try:
            # 1. 寻找按钮
            print(">>> 正在寻找 [查看全部评价] 按钮...")
            # 有时候按钮在页面底部，需要先滚动主页面让按钮加载出来
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
            time.sleep(1)
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            # 使用你提供的精确 Class 定位
            wait = WebDriverWait(self.driver, 10)
            show_btn = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, BTN_SHOW_ALL_CLASS)))

            # 2. 点击按钮
            self.driver.execute_script("arguments[0].click();", show_btn)
            print(">>> 按钮点击成功，等待侧边栏弹出...")
            time.sleep(3) # 等待侧边栏动画
            return True
        except Exception as e:
            print(f"!!! 无法打开侧边栏，错误: {e}")
            # 截图方便调试
            self.driver.save_screenshot("error_open_sidebar.png")
            return False

    def parse_visible_items(self):
        """解析当前DOM中存在的评论"""
        current_data = []
        try:
            # 获取所有评论卡片
            items = self.driver.find_elements(By.CLASS_NAME, COMMENT_ITEM_CLASS)

            for item in items:
                try:
                    # 提取内容
                    try:
                        content_el = item.find_element(By.CLASS_NAME, CONTENT_CLASS)
                        content = content_el.text.strip()
                    except:
                        content = "无文本内容"

                    # 提取元数据 (时间/SKU)
                    try:
                        meta_el = item.find_element(By.CLASS_NAME, META_CLASS)
                        meta_text = meta_el.text.replace('\n', ' ').strip()
                    except:
                        meta_text = ""

                    # 提取用户名
                    try:
                        user_el = item.find_element(By.CLASS_NAME, USER_NAME_CLASS)
                        user_name = user_el.text.strip()
                    except:
                        user_name = "匿名"

                    # 生成指纹用于去重 (内容+用户+时间)
                    unique_str = f"{user_name}{meta_text}{content}"
                    item_hash = hashlib.md5(unique_str.encode('utf-8')).hexdigest()

                    if item_hash not in self.scraped_hashes:
                        self.scraped_hashes.add(item_hash)
                        current_data.append({
                            "user": user_name,
                            "meta": meta_text,
                            "content": content,
                            "scrape_time": time.strftime("%Y-%m-%d %H:%M:%S")
                        })
                except Exception as inner_e:
                    continue

        except Exception as e:
            print(f"解析过程出错: {e}")

        return current_data, items

    def scroll_sidebar(self):
        """
        侧边栏滚动核心逻辑：
        由于侧边栏是悬浮层，window.scrollTo 无效。
        我们需要找到最后一条评论，并将其滚动到视图中，从而触发懒加载。
        """
        MAX_SCROLL_ATTEMPTS = 50 # 最大滚动次数，防止死循环
        no_new_data_count = 0    # 连续未获取新数据的次数

        # 写入CSV头
        with open(CSV_FILE, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=["user", "meta", "content", "scrape_time"])
            writer.writeheader()

        for i in range(MAX_SCROLL_ATTEMPTS):
            print(f"--- 滚动第 {i+1} 次 ---")

            # 1. 解析当前数据
            new_items, all_elements = self.parse_visible_items()

            # 2. 存入 CSV
            if new_items:
                print(f"新增 {len(new_items)} 条数据")
                no_new_data_count = 0
                with open(CSV_FILE, 'a', encoding='utf-8-sig', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=["user", "meta", "content", "scrape_time"])
                    writer.writerows(new_items)
            else:
                print("本轮未发现新数据...")
                no_new_data_count += 1

            # 3. 终止条件：连续3次滚动都没有新数据，认为到底了
            if no_new_data_count >= 3:
                print(">>> 连续多次未检测到新评论，判定已爬取完毕。")
                break

            # 4. 执行滚动操作
            if all_elements:
                last_element = all_elements[-1]
                # 关键：将最后一个元素滚动到可视区域
                self.driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'end'});", last_element)
            else:
                print("未找到评论元素，无法滚动")
                break

            # 5. 随机等待，模拟真人浏览并等待网络加载
            time.sleep(random.uniform(2, 4))

    def run(self):
        try:
            self.login_phase()
            if self.open_sidebar():
                self.scroll_sidebar()
            print(f">>> 任务结束，共去重抓取 {len(self.scraped_hashes)} 条评论。")
            print(f"数据已保存至: {CSV_FILE}")
        finally:
            # 调试阶段可以注释掉 quit，保留浏览器查看现场
            # self.driver.quit()
            pass

if __name__ == "__main__":
    spider = TmallSidebarSpider()
    spider.run()

>>> 打开登录页，请在 40秒内 扫码登录...
>>> 正在访问商品页: https://detail.tmall.com/item.htm?abbucket=17&id=903091730827...
>>> 正在寻找 [查看全部评价] 按钮...
!!! 无法打开侧边栏，错误: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=143.0.7499.146); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x701223
	0x701264
	0x4ee6dd
	0x4dd99e
	0x4fc4e6
	0x5621d8
	0x57d44c
	0x55b2e6
	0x52d321
	0x52e1d4
	0x955264
	0x95081b
	0x96d0fa
	0x71b128
	0x72312d
	0x709528
	0x7096e9
	0x6f3a78
	0x7690fcc9
	0x771582ae
	0x7715827e



InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x701223
	0x701264
	0x4ee52b
	0x52c635
	0x55b3a6
	0x556ed1
	0x556846
	0x4bebfd
	0x4bf18e
	0x4bf65d
	0x955264
	0x95081b
	0x96d0fa
	0x71b128
	0x72312d
	0x4be7ab
	0x4bddf7
	0xaa16ff
	0x7690fcc9
	0x771582ae
	0x7715827e


In [17]:
import time
import csv
import random
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# ================= 配置区域 =================
#TARGET_URL = "https://detail.tmall.com/item.htm?abbucket=17&id=903091730827"
TARGET_URL = "https://detail.tmall.com/item.htm?abbucket=17&id=775862636421"
CSV_FILE = 'tmall_sidebar_reviews_v2.csv'
MAX_COMMENTS = 5000

class TmallProSpider:
    def __init__(self):
        self.driver = self._init_driver()

    def _init_driver(self):
        options = Options()
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument('--disable-blink-features=AutomationControlled')
        # 窗口尽量大，保证侧边栏完整显示
        options.add_argument('--start-maximized')

        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

        driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
            "source": """
            Object.defineProperty(navigator, 'webdriver', { get: () => undefined })
            """
        })
        return driver

    def login(self):
        print(">>> [步骤1] 打开登录页...")
        self.driver.get("https://login.taobao.com/member/login.jhtml")
        input(">>> 请在浏览器扫码登录，登录成功跳转后，请在此终端按【回车】继续...")

    def open_sidebar(self):
        print(f">>> [步骤2] 进入商品页: {TARGET_URL}")
        self.driver.get(TARGET_URL)
        time.sleep(3)

        try:
            print(">>> 正在定位 [查看全部评价] 按钮...")
            # 显式等待按钮加载
            btn = WebDriverWait(self.driver, 15).until(
                EC.element_to_be_clickable((By.CLASS_NAME, "ShowButton--fMu7HZNs"))
            )
            # 将按钮滚动到屏幕中间防止被遮挡
            self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
            time.sleep(1.5)
            btn.click()
            print(">>> 按钮已点击，等待侧边栏弹出...")

            # 等待第一条评论出现，确保侧边栏DOM已生成
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.CLASS_NAME, "Comment--H5QmJwe9"))
            )
            return True
        except Exception as e:
            print(f"!!! 打开侧边栏失败: {e}")
            return False

    def human_scroll(self, element_container):
        """
        模拟真人拉动滚动条：下拉 -> 回弹 -> 再拉到底
        element_container: 评论列表的父容器
        """
        try:
            # 1. 猛力向下拉到底
            self.driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", element_container)
            time.sleep(random.uniform(0.5, 1.0))

            # 2. 模拟“看漏了”，向上回滚一点点 (300-500像素)
            self.driver.execute_script("arguments[0].scrollTop = arguments[0].scrollTop - 500;", element_container)
            time.sleep(random.uniform(0.8, 1.5)) # 停留看一下

            # 3. 再次触底，触发加载
            self.driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", element_container)
            time.sleep(random.uniform(1.5, 2.5)) # 等待加载转圈结束

        except Exception as e:
            print(f"滚动异常: {e}")

    def scrape_logic(self):
        unique_set = set()
        scraped_count = 0
        no_change_times = 0

        print(">>> [步骤3] 开始沉浸式爬取...")

        while scraped_count < MAX_COMMENTS:
            # 1. 重新获取所有评论元素
            comments = self.driver.find_elements(By.CLASS_NAME, "Comment--H5QmJwe9")

            if not comments:
                print("未找到评论元素，可能页面未加载或类名变化")
                break

            # 【核心逻辑】找到这些评论的“父容器”
            # XPath "./.." 表示选取当前节点的父节点
            # 因为评论都在一个 div 里，只要找到其中一个评论的父节点，就是那个可以滚动的容器
            scroll_container = comments[-1].find_element(By.XPATH, "./..")

            # 2. 遍历并解析（只处理新数据）
            current_batch_new = []
            for item in comments:
                # 简单解析内容用于去重
                try:
                    raw_text = item.text
                    content_hash = hash(raw_text)

                    if content_hash not in unique_set:
                        # === 增加停留时间，模拟阅读每条评论 ===
                        time.sleep(random.uniform(0.1, 0.3))

                        data = self.parse_single_comment(item)
                        if data:
                            current_batch_new.append(data)
                            unique_set.add(content_hash)
                except:
                    continue

            # 3. 保存与反馈
            if current_batch_new:
                self.save_to_csv(current_batch_new, scraped_count == 0)
                scraped_count += len(current_batch_new)
                print(f"已获取: {scraped_count} 条 | 本次新增: {len(current_batch_new)} 条")
                no_change_times = 0 # 重置计数器
            else:
                no_change_times += 1
                print(f"等待加载更多... ({no_change_times}/5)")

            # 4. 判断结束条件
            if no_change_times >= 5:
                print(">>> 多次滚动未发现新内容，判定已到底。")
                break

            # 5. 执行“拉锯式”滚动加载
            self.human_scroll(scroll_container)

    def parse_single_comment(self, element):
        """解析单条详情"""
        try:
            # 使用 CSS Selector 在当前 element 内部查找
            content = element.find_element(By.CLASS_NAME, "content--uonoOhaz").text.strip()

            try:
                nickname = element.find_element(By.CLASS_NAME, "userName--KpyzGX2s").text.split('\n')[0]
            except:
                nickname = "匿名"

            try:
                date_str = element.find_element(By.CLASS_NAME, "meta--PLijz6qf").text
            except:
                date_str = ""

            return {
                "nickname": nickname,
                "content": content,
                "date_meta": date_str
            }
        except:
            return None

    def save_to_csv(self, data, is_first):
        if not data: return
        mode = 'w' if is_first else 'a'
        with open(CSV_FILE, mode, encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=["nickname", "content", "date_meta"])
            if is_first: writer.writeheader()
            writer.writerows(data)

    def run(self):
        try:
            self.login()
            if self.open_sidebar():
                self.scrape_logic()
        finally:
            print(">>> 程序结束")
            # self.driver.quit() # 注释掉这一行，方便结束后你查看浏览器状态

if __name__ == "__main__":
    spider = TmallProSpider()
    spider.run()

>>> [步骤1] 打开登录页...
>>> [步骤2] 进入商品页: https://detail.tmall.com/item.htm?abbucket=17&id=775862636421
>>> 程序结束


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=143.0.7499.146); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x701223
	0x701264
	0x4ee6dd
	0x4dd99e
	0x4fc4e6
	0x5621d8
	0x57d44c
	0x55b2e6
	0x52d321
	0x52e1d4
	0x955264
	0x95081b
	0x96d0fa
	0x71b128
	0x72312d
	0x709528
	0x7096e9
	0x6f3a78
	0x7690fcc9
	0x771582ae
	0x7715827e


In [ ]:
#一次登录验证实现多页面评论爬取

In [3]:
import time
import csv
import random
import re
import os
import sys
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ================= 配置区域 =================
INPUT_EXCEL = '血糖仪.xlsx'  # 必须确保这个文件在同级目录
MAX_COMMENTS_PER_ITEM = 60
# ===========================================

class TmallBatchSpider:
    def __init__(self):
        # 在启动浏览器前，先检查Excel文件是否存在
        if not os.path.exists(INPUT_EXCEL):
            print(f"\n[严重错误] 找不到文件: {INPUT_EXCEL}")
            print(f"请确保创建了一个Excel文件，命名为 '{INPUT_EXCEL}'，并放在代码所在的同一个文件夹中。")
            print("Excel的第一行第一列必须写 'url'，下面填入商品链接。")
            input(">>> 按回车键退出程序...")
            sys.exit()

        print(">>> 环境检查通过，正在启动浏览器...")
        self.driver = self._init_driver()

    def _init_driver(self):
        options = Options()
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_argument('--start-maximized')

        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
            "source": "Object.defineProperty(navigator, 'webdriver', { get: () => undefined })"
        })
        return driver

    def extract_item_id(self, url):
        try:
            match = re.search(r'id=(\d+)', url)
            return match.group(1) if match else "unknown"
        except:
            return "unknown"

    def login_phase(self):
        """更智能的登录检测"""
        print("\n" + "="*50)
        print(">>> [步骤 1] 正在跳转登录页...")
        self.driver.get("https://login.taobao.com/member/login.jhtml")

        print(">>> 请在浏览器上完成扫码。")
        print(">>> 扫码成功后，页面通常会自动跳转回淘宝首页。")

        # 循环检测机制：等待用户按回车，或者检测URL变化
        while True:
            choice = input(">>> 扫码登录完成后，请务必【点击此黑窗口】并按回车确认: ")

            # 简单检查当前URL，如果是登录页，提示用户可能没成功
            current_url = self.driver.current_url
            if "login.taobao.com" in current_url:
                print("!!! 检测到您似乎还在登录页面。")
                confirm = input("    确认已经扫码成功了吗？(y/n): ")
                if confirm.lower() == 'y':
                    break
            else:
                print(">>> 检测到页面已跳转，登录成功！")
                break

        print("="*50 + "\n")

    def open_sidebar(self, url):
        print(f"   >>> 打开链接: {url[:60]}...")
        self.driver.get(url)
        time.sleep(3)

        try:
            # 查找查看全部评价按钮
            btn = WebDriverWait(self.driver, 8).until(
                EC.element_to_be_clickable((By.CLASS_NAME, "ShowButton--fMu7HZNs"))
            )
            self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
            time.sleep(1)
            btn.click()

            WebDriverWait(self.driver, 8).until(
                EC.presence_of_element_located((By.CLASS_NAME, "Comment--H5QmJwe9"))
            )
            return True
        except Exception as e:
            print(f"   [跳过] 无法打开侧边栏 (可能无评论或需验证): {str(e)[:50]}")
            return False

    def human_scroll(self, container):
        try:
            self.driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", container)
            time.sleep(random.uniform(1, 0.8))
            self.driver.execute_script("arguments[0].scrollTop = arguments[0].scrollTop - 200;", container)
            time.sleep(random.uniform(1, 0.8))
            self.driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", container)
            time.sleep(random.uniform(2, 1.5))
        except:
            pass

    def process_item(self, url):
        item_id = self.extract_item_id(url)
        output_file = f'result_{item_id}.csv'

        if not self.open_sidebar(url): return

        unique_set = set()
        scraped = 0
        no_change = 0

        # 打开文件准备写入
        with open(output_file, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=["Nickname", "Content", "Meta"])
            writer.writeheader()

            while scraped < MAX_COMMENTS_PER_ITEM:
                comments = self.driver.find_elements(By.CLASS_NAME, "Comment--H5QmJwe9")
                if not comments: break

                scroll_container = comments[-1].find_element(By.XPATH, "./..")

                new_data_count = 0
                for item in comments:
                    try:
                        raw = item.text
                        if hash(raw) in unique_set: continue

                        content = item.find_element(By.CLASS_NAME, "content--uonoOhaz").text.strip()
                        nickname = item.find_element(By.CLASS_NAME, "userName--KpyzGX2s").text.split('\n')[0]
                        meta = item.find_element(By.CLASS_NAME, "meta--PLijz6qf").text

                        writer.writerow({"Nickname": nickname, "Content": content, "Meta": meta})
                        unique_set.add(hash(raw))
                        scraped += 1
                        new_data_count += 1
                    except: continue

                if new_data_count > 0:
                    print(f"   进度: {scraped} 条 (新增 {new_data_count})")
                    no_change = 0
                else:
                    no_change += 1
                    if no_change >= 4: break # 减少等待次数加快速度

                self.human_scroll(scroll_container)

        print(f"   [完成] 数据已保存至 {output_file}")

    def run(self):
        try:
            # 1. 登录
            self.login_phase()

            # 2. 读取Excel
            print(f">>> 正在读取 {INPUT_EXCEL}...")
            try:
                df = pd.read_excel(INPUT_EXCEL)
                # 兼容大小写，去空格
                df.columns = [c.strip().lower() for c in df.columns]
                if 'url' not in df.columns:
                    raise ValueError("Excel中找不到 'url' 这一列 (请检查表头)")
                urls = df['url'].dropna().tolist()
                print(f">>> 成功加载 {len(urls)} 个任务。")
            except Exception as e:
                print(f"\n[Excel读取错误] {e}")
                print("请检查：1.文件名是否正确 2.文件是否已关闭 3.表头是否包含url")
                return

            # 3. 执行
            for i, url in enumerate(urls):
                print(f"\n>>> [任务 {i+1}/{len(urls)}] 处理中...")
                self.process_item(url)

                if i < len(urls) - 1:
                    wait = random.randint(10, 10)
                    print(f">>> 休息 {wait} 秒...")
                    time.sleep(wait)

            print("\n>>> 所有任务执行完毕！")

        except Exception as e:
            print(f"\n[程序崩溃] 发生未知错误: {e}")
            import traceback
            traceback.print_exc()
        finally:
            input("\n>>> 程序运行结束，按回车键关闭浏览器并退出...")
            try:
                self.driver.quit()
            except:
                pass

if __name__ == "__main__":
    spider = TmallBatchSpider()
    spider.run()

>>> 环境检查通过，正在启动浏览器...

>>> [步骤 1] 正在跳转登录页...
>>> 请在浏览器上完成扫码。
>>> 扫码成功后，页面通常会自动跳转回淘宝首页。
>>> 检测到页面已跳转，登录成功！

>>> 正在读取 血糖仪.xlsx...
>>> 成功加载 3 个任务。

>>> [任务 1/3] 处理中...
   >>> 打开链接: https://detail.tmall.com/item.htm?abbucket=17&id=93408226335...
   进度: 2 条 (新增 2)
   进度: 22 条 (新增 20)
   进度: 62 条 (新增 40)
   [完成] 数据已保存至 result_934082263359.csv
>>> 休息 10 秒...

>>> [任务 2/3] 处理中...
   >>> 打开链接: https://detail.tmall.com/item.htm?abbucket=17&id=83460782390...
   进度: 2 条 (新增 2)
   进度: 22 条 (新增 20)
   进度: 62 条 (新增 40)
   [完成] 数据已保存至 result_834607823900.csv
>>> 休息 10 秒...

>>> [任务 3/3] 处理中...
   >>> 打开链接: https://detail.tmall.com/item.htm?abbucket=17&id=78776940469...
   进度: 2 条 (新增 2)
   进度: 22 条 (新增 20)
   进度: 62 条 (新增 40)
   [完成] 数据已保存至 result_787769404691.csv

>>> 所有任务执行完毕！


In [32]:
import pandas as pd
import os
import re
import jieba
import torch
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm  # 进度条库

# ================= 配置区域 =================
# 文件夹路径 ('.' 代表当前代码所在文件夹)
FOLDER_PATH = 'D:\\study\\spyder'
# 中文停用词表 (可以自己扩展，这里提供基础版)
STOP_WORDS = {
    '的', '了', '在', '是', '我', '有', '和', '就', '不', '人', '都', '一', '一个', '上', '也', '很', '到', '说', '要', '去', '你',
    '会', '着', '没有', '看', '好', '自己', '这', '那', '回复', '即使', '其实', '因为', '还是', '非常', '那个', '这个', '啊', '哦',
    '嘛', '吧', '但是', '但', '虽然', '只是', '不过', '买', '购', '觉得', '感觉', '收到'
}
# 词云字体路径 (Windows默认微软雅黑，Mac需改为 '/System/Library/Fonts/PingFang.ttc')
FONT_PATH = 'C:\\Windows\\Fonts\\msyh.ttc'
# ===========================================

def clean_meta_data(row):
    """
    处理 Meta 列：提取日期 和 SKU
    输入示例: "2025年12月13日已购：【二盒 】..." 或 "2025-11-30已购..."
    """
    meta_text = str(row)

    # 1. 提取日期 (兼容 YYYY年MM月DD日 和 YYYY-MM-DD)
    # 正则逻辑：匹配4位数字 + (年|-) + 1-2位数字 + (月|-) + 1-2位数字 + (日|空)
    date_pattern = re.search(r'(\d{4})[年-](\d{1,2})[月-](\d{1,2})[日]?', meta_text)

    formatted_date = ""
    if date_pattern:
        year, month, day = date_pattern.groups()
        # 格式化为 YYYY-MM-DD
        formatted_date = f"{year}-{int(month):02d}-{int(day):02d}"

    # 2. 提取 SKU (截取 '已购：' 后面的内容)
    sku_info = ""
    if '已购：' in meta_text:
        sku_info = meta_text.split('已购：')[1].strip()
    elif '已购:' in meta_text: # 兼容英文冒号
        sku_info = meta_text.split('已购:')[1].strip()

    return formatted_date, sku_info

def analyze_sentiment_bert(texts):
    """
    使用 BERT 模型进行批量情感分析
    使用模型: uer/roberta-base-finetuned-jd-binary-chinese (基于京东评论微调的模型，非常适合电商场景)
    """
    print(">>> 正在加载 BERT 情感分析模型 (首次运行会自动下载约400M模型)...")

    # 检测是否有 GPU，如果有则使用 GPU 加速
    device = 0 if torch.cuda.is_available() else -1

    # 加载 Hugging Face 的 Pipeline
    # 这是一个二分类模型：label_1 (正面/积极), label_0 (负面/消极)
    sentiment_pipeline = pipeline(
        'sentiment-analysis',
        model='uer/roberta-base-finetuned-jd-binary-chinese',
        device=device
    )

    results = []
    print(f">>> 开始分析 {len(texts)} 条评论的情感...")

    # 批量处理防止内存溢出，且带进度条
    for text in tqdm(texts, desc="BERT Analysis"):
        # BERT模型通常限制512个token，截断处理防止报错
        truncated_text = text[:500]
        try:
            res = sentiment_pipeline(truncated_text)[0]
            # 转换标签为中文便于阅读
            label_cn = "积极" if res['label'] == 'positive (stars 4, 5)' or res['label'] == 'label_1' else "消极"
            # 如果模型输出是 label_1/label_0 格式
            if res['label'] == 'label_1': label_cn = '积极'
            elif res['label'] == 'label_0': label_cn = '消极'

            results.append({
                'sentiment_label': label_cn,
                'sentiment_score': round(res['score'], 4)
            })
        except Exception as e:
            results.append({'sentiment_label': 'Error', 'sentiment_score': 0})

    return results

def generate_word_cloud(text_list, file_name_prefix):
    """生成词云图并保存"""
    full_text = " ".join([str(t) for t in text_list])

    # 分词
    words = jieba.cut(full_text)
    cleaned_words = []
    for word in words:
        if len(word) > 1 and word not in STOP_WORDS:
            cleaned_words.append(word)

    # 统计 Top 100
    word_counts = Counter(cleaned_words)
    top_100 = word_counts.most_common(100)
    print(f"\n>>> [{file_name_prefix}] Top 10 关键词预览: {top_100[:10]}")

    # 生成词云
    # 如果没有中文字体，词云会显示乱码方框，请务必确认 FONT_PATH
    if not os.path.exists(FONT_PATH):
        print(f"!!! 警告：未找到字体文件 {FONT_PATH}，词云可能显示乱码。请修改代码中的 FONT_PATH。")

    wc = WordCloud(
        font_path=FONT_PATH,
        width=1000, height=700,
        background_color='white',
        max_words=100
    ).generate_from_frequencies(word_counts)

    # 保存图片
    pic_name = f"{file_name_prefix}_wordcloud.png"
    wc.to_file(pic_name)
    print(f">>> 词云图已保存为: {pic_name}")

def process_files():
    # 获取所有xlsx文件
    files = [f for f in os.listdir(FOLDER_PATH) if f.endswith('.xlsx') and not f.startswith('processed_')]

    if not files:
        print("未找到 .xlsx 文件！")
        return

    for file in files:
        print(f"\n{'='*20} 正在处理文件: {file} {'='*20}")

        # 1. 读取数据
        try:
            df = pd.read_excel(os.path.join(FOLDER_PATH, file))
        except Exception as e:
            print(f"读取失败: {e}")
            continue

        # 检查列名是否对应
        required_cols = ['Nickname', 'Content', 'Meta']
        if not all(col in df.columns for col in required_cols):
            print(f"跳过文件 {file}，列名不匹配。需要: {required_cols}")
            continue

        # 2. 处理 Meta 列 (日期与SKU)
        # 使用 apply 应用函数，返回 Series 分别赋值
        extracted_data = df['Meta'].apply(clean_meta_data)
        df['Date'] = [x[0] for x in extracted_data]
        df['SKU'] = [x[1] for x in extracted_data]

        # 3. BERT 情感分析
        # 获取所有 content 内容转为列表
        contents = df['Content'].astype(str).tolist()
        sentiment_results = analyze_sentiment_bert(contents)

        df['Sentiment'] = [x['sentiment_label'] for x in sentiment_results]
        df['Sentiment_Score'] = [x['sentiment_score'] for x in sentiment_results]

        # 4. 生成词云
        generate_word_cloud(contents, file.replace('.xlsx', ''))

        # 5. 保存结果
        output_filename = f"processed_{file}"
        # 调整列顺序，好看一点
        output_cols = ['Nickname', 'Date', 'SKU', 'Content', 'Sentiment', 'Sentiment_Score', 'Meta']
        df = df[output_cols]

        df.to_excel(output_filename, index=False)
        print(f">>> 处理完成！结果已保存至: {output_filename}")

if __name__ == "__main__":
    process_files()


==================== 正在处理文件: 【新品】鱼跃动态血糖仪.xlsx ====================
>>> 正在加载 BERT 情感分析模型 (首次运行会自动下载约400M模型)...


Device set to use cpu


>>> 开始分析 1435 条评论的情感...


BERT Analysis: 100%|██████████| 1435/1435 [01:13<00:00, 19.55it/s]



>>> [【新品】鱼跃动态血糖仪] Top 10 关键词预览: [('血糖', 544), ('方便', 416), ('可以', 293), ('血糖仪', 181), ('数据', 178), ('不错', 168), ('监测', 167), ('使用', 166), ('佩戴', 157), ('动态', 151)]
>>> 词云图已保存为: 【新品】鱼跃动态血糖仪_wordcloud.png
>>> 处理完成！结果已保存至: processed_【新品】鱼跃动态血糖仪.xlsx

==================== 正在处理文件: 【瞬感2代】雅培动态血糖仪.xlsx ====================
>>> 正在加载 BERT 情感分析模型 (首次运行会自动下载约400M模型)...


...


>>> [硅基轻享动态血糖仪] Top 10 关键词预览: [('血糖', 924), ('方便', 631), ('可以', 519), ('动态', 339), ('不错', 289), ('血糖仪', 280), ('使用', 279), ('监测', 275), ('客服', 266), ('佩戴', 248)]
>>> 词云图已保存为: 硅基轻享动态血糖仪_wordcloud.png
>>> 处理完成！结果已保存至: processed_硅基轻享动态血糖仪.xlsx

==================== 正在处理文件: 诺芯无创血糖仪免扎针家用测试.xlsx ====================
>>> 正在加载 BERT 情感分析模型 (首次运行会自动下载约400M模型)...


Device set to use cpu


>>> 开始分析 49 条评论的情感...


BERT Analysis: 100%|██████████| 49/49 [00:03<00:00, 15.56it/s]



>>> [诺芯无创血糖仪免扎针家用测试] Top 10 关键词预览: [('血糖仪', 27), ('好好', 22), ('血糖', 20), ('操作', 16), ('使用', 15), ('无创', 14), ('手指', 13), ('标定', 12), ('检测', 12), ('扎针', 12)]
>>> 词云图已保存为: 诺芯无创血糖仪免扎针家用测试_wordcloud.png
>>> 处理完成！结果已保存至: processed_诺芯无创血糖仪免扎针家用测试.xlsx
